In [ ]:
import torch
import torch.nn as nn
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from torch.optim.lr_scheduler import StepLR, ReduceLROnPlateau


In [ ]:
path = "/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/final_data"
unseen_path = "/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/Autoencoder/unseen_data/unseen_data.csv"

euler_nr = pd.read_csv(os.path.join(path, "euler_nr_data.csv"))
datatable = pd.read_csv(os.path.join(path, "Final_data.csv"))

euler_nr = euler_nr[['SubjectID', 'avg_euler_centered_neg_sqrt']].copy()
df = datatable.merge(euler_nr, how="right")
df = df.iloc[:, 2:]

#create dataframe without the last few columns
df_for_blr = df.iloc[:,0:112]
df_unseen = df_for_blr[:3000]
# df_unseen.to_csv(unseen_path)
df_for_blr = df_for_blr[3000:]

#these cols had only zeros (left-vessel had 0s in some cases but not all --> may fit a separate model for this)
df_for_blr = df_for_blr.drop(columns=['SubjectID','5th-Ventricle','Left-WM-hypointensities','Left-non-WM-hypointensities','Left-vessel','Left-WM-hypointensities','Left-non-WM-hypointensities','Right-WM-hypointensities','Right-non-WM-hypointensities'])
df_for_blr 

train_df, test_df = train_test_split(df_for_blr, test_size=0.2, random_state=1107)

scaler = RobustScaler()
train_scaled = scaler.fit_transform(train_df)
test_scaled = scaler.transform(test_df)

train_tensor = torch.tensor(train_scaled, dtype=torch.float32)
test_tensor = torch.tensor(test_scaled, dtype=torch.float32)

training_data = TensorDataset(train_tensor)
test_data = TensorDataset(test_tensor)

train_loader = DataLoader(training_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

In [ ]:
class AE_deep(nn.Module):
    def __init__(self, input_size=108, latent_size=10):
        super(AE_deep, self).__init__()
        self.input_size = input_size
        self.latent_size = latent_size

        # encoder network
        self.encoder = nn.Sequential(
            nn.Linear(input_size, 56), nn.ReLU(),
            nn.Linear(56, 56), nn.ReLU(),
            nn.Linear(56, 28), nn.ReLU(),
            nn.Linear(28, latent_size)
        )
        # decoder network
        self.decoder = nn.Sequential(
            nn.Linear(latent_size, 28), nn.ReLU(),
            nn.Linear(28, 56), nn.ReLU(),
            nn.Linear(56, 56), nn.ReLU(),
            nn.Linear(56, input_size)
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        z = self.encode(x)
        return self.decode(z)
        


In [ ]:
#ReduceLROnPlateau implementation
def train_ae(model, train_loader, test_loader, epochs, lr):
    loss_func = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = ReduceLROnPlateau(optimizer, mode = 'min', factor = 0.1, patience = 5)
    train_losses = []
    test_losses = []
    

    for epoch in range(epochs):
        
        ################
        #### TRAIN #####
        ################
        
        model.train()
        train_loss = 0.0
        
        for batch in train_loader:
            optimizer.zero_grad()
            x = batch[0]
            prediction = model(x)
            loss = loss_func(prediction, x)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss_average = train_loss / len(train_loader)
        train_losses.append(train_loss_average)

        
        ################
        ####  TEST #####
        ################
        
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for x, in test_loader:
                prediction = model(x)
                loss = loss_func(prediction, x)
                test_loss += loss.item()
        test_loss_average = test_loss / len(test_loader)
        test_losses.append(test_loss_average)

        scheduler.step(test_loss_average)
        print(f'Epoch: {epoch+1}')
        print(f'Train loss: {train_loss_average}')
        print(f'Test loss: {test_loss_average}')
        print('\n')

    return train_losses, test_losses
                

In [ ]:
def plot_losses(train_losses, test_losses):
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Training Loss')
    plt.plot(test_losses, label='Test Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss (MSE)')
    plt.title('Training and Test Loss')
    plt.legend()
    plt.show()


lr = 0.001
epochs = 100
input_size = df_for_blr.shape[1]
model = AE_deep(input_size=input_size, latent_size=10)

train_losses, test_losses = train_ae(model, train_loader, test_loader, epochs=epochs, lr=lr)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def evaluate_reconstruction(model, dataloader, device):
    model.eval()
    all_outputs = []
    all_inputs = []

    with torch.no_grad():
        for batch in dataloader:
            x, = batch
            x = x.to(device)
            output = model(x)
            all_outputs.append(output.cpu().numpy())
            all_inputs.append(x.cpu().numpy())

    all_outputs = np.concatenate(all_outputs, axis=0)
    all_inputs = np.concatenate(all_inputs, axis=0)

    mse = mean_squared_error(all_inputs, all_outputs)
    mae = mean_absolute_error(all_inputs, all_outputs)
    r2 = r2_score(all_inputs, all_outputs)

    print(f"MSE: {mse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"R² Score: {r2:.4f}")

    return mse, mae, r2

mse, mae, r2 = evaluate_reconstruction(model, test_loader, device)
plot_losses(train_losses, test_losses)

**the depth of the autoencoder does not significantly alter performance**